In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
experiment = "cross-session_7-channels"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]

datasets = [ "TTK", "Zhou2016", "BNCI2014_001", "Wang2025", "Yang2025_3C", "Zhou2020" ]


In [4]:
for pipeline in [*mi_pipelines.values(), *ood_pipelines.values()]:
    channel_params = {key: tuple(COMMON_CHANNELS) for key in pipeline.get_params(deep=True) if key.split("__")[-1] == "channel_names"}
    pipeline.set_params(**channel_params)

MI_METHODS = ["ACM+TS+SVM", "FBCSP+ElasticNet", "ShallowConvNet", "EEGNet", "Microstates+GB", "VAR+shLDA", "Log-bandpower+shLDA"]
OOD_METHODS = ["ACM+TS+OCSVM", "FBCSP+IsolationForest", "ShallowConvNet+kNN", "EEGNet+TempDens", "Log-bandpower+Mahalanobis", "Microstates+LOF", "VAR+Mahalanobis"]

In [5]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None

        exp_type = "mi"
        for method, pipeline in mi_pipelines.items():
            if method not in MI_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": exp_type, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        exp_type = "ood"
        data = None
        for method, pipeline in ood_pipelines.items():
            if method not in OOD_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": exp_type, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                ood=("feet",)
                if dataset == "TTK":
                    ood = ("left_foot", "right_foot")
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, ood_classes=ood )
                train_data, val_data, test_data = data
            model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood", metric="OOD_AUROC")


TTK


,MI_BA,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
EEGNet,0.633000,6,0.740000,0.620000,0.600000,0.600000,0.716000,0.520000
FBCSP+ElasticNet,0.626000,6,0.890000,0.530000,0.540000,0.530000,0.695000,0.570000
ACM+TS+SVM,0.624000,6,0.840000,0.600000,0.500000,0.560000,0.746000,0.500000
Log-bandpower+shLDA,0.601000,6,0.730000,0.600000,0.540000,0.540000,0.629000,0.570000
Microstates+GB,0.593000,6,0.610000,0.580000,0.530000,0.600000,0.628000,0.610000
ShallowConvNet,0.589000,6,0.710000,0.630000,0.500000,0.490000,0.637000,0.570000
VAR+shLDA,0.578000,6,0.640000,0.610000,0.530000,0.550000,0.637000,0.500000


,MI_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ACM+TS+SVM,0.731000,6,0.922000,0.673000,0.626000,0.646000,0.864000,0.652000
FBCSP+ElasticNet,0.708000,6,0.988000,0.574000,0.711000,0.504000,0.844000,0.631000
EEGNet,0.686000,6,0.926000,0.597000,0.608000,0.626000,0.818000,0.544000
Log-bandpower+shLDA,0.652000,6,0.848000,0.568000,0.593000,0.527000,0.791000,0.587000
ShallowConvNet,0.639000,6,0.888000,0.632000,0.528000,0.514000,0.683000,0.592000
VAR+shLDA,0.623000,6,0.700000,0.612000,0.685000,0.580000,0.648000,0.515000
Microstates+GB,0.611000,6,0.732000,0.542000,0.504000,0.614000,0.652000,0.622000


,OOD_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
FBCSP+IsolationForest,0.780000,6,0.858000,0.825000,0.671000,0.778000,0.856000,0.690000
ShallowConvNet+kNN,0.779000,6,0.906000,0.684000,0.758000,0.869000,0.813000,0.644000
EEGNet+TempDens,0.757000,6,0.877000,0.690000,0.690000,0.719000,0.828000,0.739000
Log-bandpower+Mahalanobis,0.627000,6,0.660000,0.738000,0.570000,0.656000,0.600000,0.541000
VAR+Mahalanobis,0.626000,6,0.635000,0.717000,0.604000,0.561000,0.596000,0.647000
ACM+TS+OCSVM,0.571000,6,0.568000,0.548000,0.535000,0.614000,0.614000,0.544000
Microstates+LOF,0.565000,6,0.669000,0.544000,0.575000,0.657000,0.504000,0.440000


Zhou2016


,MI_BA,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet,0.873000,4,0.840000,0.822000,0.910000,0.920000
FBCSP+ElasticNet,0.821000,4,0.750000,0.722000,0.890000,0.920000
ShallowConvNet,0.820000,4,0.740000,0.711000,0.910000,0.920000
Microstates+GB,0.767000,4,0.680000,0.667000,0.810000,0.910000
ACM+TS+SVM,0.741000,4,0.600000,0.633000,0.860000,0.870000
Log-bandpower+shLDA,0.734000,4,0.680000,0.667000,0.880000,0.710000
VAR+shLDA,0.663000,4,0.670000,0.633000,0.610000,0.740000


... (14 outputs left out)

Zhou2020


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ShallowConvNet,0.579000,20,0.517000,0.517000,0.667000,0.550000,0.483000,0.683000,0.883000,0.500000,0.525000,0.567000,0.625000,0.492000,0.583000,0.617000,0.775000,0.508000,0.492000,0.550000,0.517000,0.533000
EEGNet,0.576000,20,0.492000,0.483000,0.667000,0.500000,0.517000,0.750000,0.850000,0.550000,0.508000,0.492000,0.650000,0.475000,0.533000,0.617000,0.825000,0.550000,0.483000,0.525000,0.533000,0.517000
FBCSP+ElasticNet,0.562000,20,0.500000,0.542000,0.617000,0.433000,0.567000,0.558000,0.792000,0.458000,0.558000,0.525000,0.708000,0.492000,0.500000,0.600000,0.583000,0.533000,0.550000,0.617000,0.608000,0.492000
Log-bandpower+shLDA,0.554000,20,0.542000,0.508000,0.633000,0.467000,0.475000,0.533000,0.717000,0.492000,0.517000,0.558000,0.558000,0.542000,0.575000,0.675000,0.642000,0.500000,0.517000,0.567000,0.533000,0.525000
Microstates+GB,0.550000,20,0.600000,0.583000,0.642000,0.483000,0.500000,0.675000,0.733000,0.517000,0.458000,0.442000,0.475000,0.567000,0.575000,0.483000,0.683000,0.442000,0.550000,0.508000,0.517000,0.567000
ACM+TS+SVM,0.530000,20,0.500000,0.475000,0.658000,0.492000,0.500000,0.500000,0.658000,0.500000,0.558000,0.500000,0.525000,0.500000,0.500000,0.642000,0.517000,0.500000,0.500000,0.567000,0.508000,0.500000
VAR+shLDA,0.526000,20,0.408000,0.517000,0.492000,0.467000,0.500000,0.600000,0.650000,0.450000,0.508000,0.542000,0.508000,0.458000,0.550000,0.575000,0.617000,0.500000,0.475000,0.558000,0.567000,0.575000


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ShallowConvNet,0.612000,20,0.520000,0.461000,0.712000,0.548000,0.535000,0.843000,0.938000,0.511000,0.564000,0.539000,0.612000,0.568000,0.608000,0.680000,0.869000,0.528000,0.493000,0.599000,0.537000,0.573000
EEGNet,0.605000,20,0.506000,0.476000,0.769000,0.467000,0.529000,0.849000,0.939000,0.553000,0.535000,0.523000,0.658000,0.522000,0.593000,0.639000,0.895000,0.572000,0.486000,0.540000,0.524000,0.528000
ACM+TS+SVM,0.602000,20,0.436000,0.507000,0.751000,0.416000,0.490000,0.784000,0.804000,0.489000,0.610000,0.463000,0.612000,0.632000,0.520000,0.683000,0.832000,0.512000,0.493000,0.664000,0.681000,0.664000
FBCSP+ElasticNet,0.602000,20,0.584000,0.550000,0.696000,0.413000,0.563000,0.771000,0.878000,0.459000,0.608000,0.580000,0.820000,0.476000,0.524000,0.683000,0.815000,0.491000,0.475000,0.662000,0.584000,0.408000
Microstates+GB,0.579000,20,0.653000,0.557000,0.696000,0.515000,0.548000,0.672000,0.838000,0.525000,0.454000,0.439000,0.469000,0.526000,0.624000,0.531000,0.786000,0.426000,0.578000,0.473000,0.553000,0.714000
Log-bandpower+shLDA,0.566000,20,0.492000,0.480000,0.657000,0.432000,0.470000,0.709000,0.804000,0.475000,0.535000,0.551000,0.570000,0.530000,0.584000,0.688000,0.715000,0.464000,0.512000,0.588000,0.516000,0.541000
VAR+shLDA,0.557000,20,0.427000,0.536000,0.556000,0.483000,0.508000,0.754000,0.762000,0.430000,0.563000,0.518000,0.538000,0.489000,0.602000,0.635000,0.642000,0.502000,0.355000,0.618000,0.577000,0.645000


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.615000,20,0.489000,0.523000,0.579000,0.618000,0.522000,0.932000,0.712000,0.511000,0.640000,0.483000,0.589000,0.619000,0.538000,0.820000,0.917000,0.647000,0.460000,0.593000,0.403000,0.710000
FBCSP+IsolationForest,0.600000,20,0.518000,0.433000,0.619000,0.516000,0.697000,0.887000,0.614000,0.496000,0.659000,0.481000,0.632000,0.541000,0.553000,0.738000,0.841000,0.635000,0.505000,0.612000,0.393000,0.636000
ShallowConvNet+kNN,0.578000,20,0.515000,0.549000,0.582000,0.603000,0.599000,0.902000,0.741000,0.480000,0.566000,0.492000,0.625000,0.573000,0.573000,0.817000,0.242000,0.655000,0.537000,0.664000,0.424000,0.427000
VAR+Mahalanobis,0.546000,20,0.556000,0.479000,0.556000,0.515000,0.630000,0.493000,0.553000,0.498000,0.587000,0.517000,0.560000,0.561000,0.510000,0.665000,0.654000,0.495000,0.479000,0.598000,0.460000,0.559000
Log-bandpower+Mahalanobis,0.539000,20,0.528000,0.441000,0.525000,0.479000,0.566000,0.418000,0.523000,0.480000,0.736000,0.493000,0.624000,0.519000,0.485000,0.661000,0.638000,0.573000,0.454000,0.580000,0.491000,0.577000
Microstates+LOF,0.530000,20,0.458000,0.546000,0.558000,0.478000,0.628000,0.747000,0.483000,0.476000,0.598000,0.550000,0.568000,0.522000,0.462000,0.676000,0.391000,0.477000,0.530000,0.369000,0.507000,0.584000
ACM+TS+OCSVM,0.517000,20,0.487000,0.420000,0.580000,0.538000,0.602000,0.643000,0.557000,0.479000,0.495000,0.466000,0.486000,0.478000,0.513000,0.574000,0.491000,0.552000,0.488000,0.487000,0.446000,0.567000


In [6]:
display_summary_by_dataset(
    experiment,
    mi_pipelines.keys(),
    ood_pipelines.keys(),
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD", "ood", "OOD_AUROC"),
    ],
)